# scPRINT2 Edge Extraction — Fix for TF→TF Dropout Bug

## What this notebook fixes

The previous extraction (cell 19 of `scprint_grn_pi_revised.ipynb`) used a column mask
that excluded all TF→TF edges:

```python
mask_edges = (dense > WEIGHT_THRESHOLD) & np.outer(~is_tf, is_tf)
#                                          ^^^^^^^
#                                          row mask requires target to be non-TF
```

Removing this mask drops every TF whose strongest outgoing edges happen to go to other TFs —
which is most of the strong-signal TFs (BATF, IRF4, FOSB, KLF2…), because master regulators
regulate each other as a core part of T cell state transitions.

**The fix:** drop the `~is_tf` row mask so TF→TF edges are kept.
Everything else (weight threshold = 0.01, self-loop removal, output schema) stays the same
so the new edge CSVs are drop-in replacements for the old ones.

## What this notebook does NOT do

- Does not rerun scPRINT2 inference. The h5ad files are fine — only the CSV extraction was bad.
- Does not change the genelist strategy. That stays as your PI's revised approach.
- Does not resolve the matrix-directionality question (rows-as-source vs cols-as-source).
  Cell 5 includes a directionality sanity check using known T-cell regulators so you can
  flag this for a follow-up if the check looks suspicious.

## Output layout

All outputs go to `.../scPRINT/grn_outputs_pi_revised_fixed/` so the original buggy CSVs
in `grn_outputs_pi_revisedv3/` remain available as a reference.

## Cell 1 — Mount Drive and imports

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

print('Imports OK.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports OK.


## Cell 2 — Paths and extraction parameters

Same `WEIGHT_THRESHOLD` as the original notebook so the only difference is the mask.

In [15]:
DRIVE_ROOT  = '/content/drive/MyDrive/benchmarking_paper/datasets'
GRN_IN_DIR  = f'{DRIVE_ROOT}/scPRINT/grn_outputs_pi_revisedv3'           # h5ad source
GRN_OUT_DIR = f'{DRIVE_ROOT}/scPRINT/grn_outputs_pi_revised_fixed'        # new CSVs
os.makedirs(GRN_OUT_DIR, exist_ok=True)

# Edge inclusion rule
WEIGHT_THRESHOLD = 0.01    # same as original cell 19
DROP_SELF_LOOPS  = True    # a TF "regulating itself" is not informative
INCLUDE_TF_TO_TF = True    # ←-- this is the bug fix; original was False

STATES = ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']

print(f'Reading h5ads from: {GRN_IN_DIR}')
print(f'Writing CSVs to:    {GRN_OUT_DIR}')
print(f'WEIGHT_THRESHOLD={WEIGHT_THRESHOLD}, '
      f'DROP_SELF_LOOPS={DROP_SELF_LOOPS}, '
      f'INCLUDE_TF_TO_TF={INCLUDE_TF_TO_TF}')

# Sanity: confirm h5ad files exist before we burn time
for state in STATES:
    path = f'{GRN_IN_DIR}/{state}_grn.h5ad'
    exists = os.path.exists(path)
    print(f'  {state}_grn.h5ad: {"found" if exists else "MISSING"}')
    if not exists:
        raise FileNotFoundError(path)

Reading h5ads from: /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3
Writing CSVs to:    /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revised_fixed
WEIGHT_THRESHOLD=0.01, DROP_SELF_LOOPS=True, INCLUDE_TF_TO_TF=True
  CD4_rest_grn.h5ad: found
  CD4_act_grn.h5ad: found
  CD8_rest_grn.h5ad: found
  CD8_act_grn.h5ad: found


## Cell 3 — Fixed edge extraction

The single line that differs from the original cell 19:

```python
# Original (buggy):
mask_edges = (dense > WEIGHT_THRESHOLD) & np.outer(~is_tf, is_tf)

# Fixed:
mask_edges = (dense > WEIGHT_THRESHOLD) & np.outer(np.ones_like(is_tf), is_tf)
#                                          ^^^^^^^^^^^^^^^^^^^^^^^^
#                                          row mask is all-True: target can be TF or non-TF
```

Everything else is identical. Output schema matches the original CSVs.

In [16]:
all_edges = {}
extraction_stats = []

for state in STATES:
    print(f'\n=== {state} ===')
    grn = ad.read_h5ad(f'{GRN_IN_DIR}/{state}_grn.h5ad')
    mat = grn.varp['GRN']
    dense = mat.toarray() if sp.issparse(mat) else np.asarray(mat)
    is_tf   = grn.var['isTF'].astype(bool).values
    symbols = grn.var['symbol'].values
    n_tfs_total = int(is_tf.sum())

    print(f'  GRN matrix: {dense.shape}, weight range [{dense.min():.5f}, {dense.max():.5f}]')
    print(f'  TFs flagged in .var[isTF]: {n_tfs_total} / {dense.shape[0]}')

    # CORRECTED ORIENTATION (diagnostic showed rows = regulators, cols = targets):
    #   - Row must be a TF (the regulator)
    #   - Column can be anything (TF or non-TF target)
    #   - Weight > threshold
    col_mask = np.ones_like(is_tf, dtype=bool) if INCLUDE_TF_TO_TF else ~is_tf
    mask_edges = (dense > WEIGHT_THRESHOLD) & np.outer(is_tf, col_mask)
    rows, cols = np.where(mask_edges)

    edges = pd.DataFrame({
        'TF':           symbols[rows],   # ROW = regulator (TF)
        'target':       symbols[cols],   # COL = target
        'weight':       dense[rows, cols],
        'target_is_tf': is_tf[cols],
        'state':        state,
    })

    if DROP_SELF_LOOPS:
        before = len(edges)
        edges = edges[edges['TF'] != edges['target']]
        print(f'  Removed {before - len(edges)} self-loops')

    edges = edges.sort_values(['TF', 'weight'], ascending=[True, False]).reset_index(drop=True)
    all_edges[state] = edges

    n_tfs        = edges['TF'].nunique()
    n_targets    = edges['target'].nunique()
    n_tf_to_tf   = int(edges['target_is_tf'].sum())
    n_tf_to_gene = len(edges) - n_tf_to_tf

    extraction_stats.append({
        'state': state,
        'tfs_in_h5ad': n_tfs_total,
        'tfs_in_csv': n_tfs,
        'n_unique_targets': n_targets,
        'n_edges_total': len(edges),
        'n_tf_to_tf_edges': n_tf_to_tf,
        'n_tf_to_nontf_edges': n_tf_to_gene,
        'fraction_tf_to_tf': round(n_tf_to_tf / len(edges), 3) if len(edges) else 0,
    })

    out_csv = f'{GRN_OUT_DIR}/{state}_scprint_pi_revised_fixed_edges.csv'
    edges.to_csv(out_csv, index=False)
    print(f'  Wrote {len(edges):,} edges ({n_tfs} TFs, {n_targets} targets)')
    print(f'    TF→TF:     {n_tf_to_tf:,}')
    print(f'    TF→non-TF: {n_tf_to_gene:,}')

    del grn, dense
    gc.collect()

combined = pd.concat(all_edges.values(), ignore_index=True)
combined_path = f'{GRN_OUT_DIR}/all_states_scprint_pi_revised_fixed_edges.csv'
combined.to_csv(combined_path, index=False)
print(f'\nCombined: {len(combined):,} total edges → {combined_path}')


=== CD4_rest ===
  GRN matrix: (4251, 4251), weight range [0.00001, 0.07992]
  TFs flagged in .var[isTF]: 333 / 4251
  Removed 7 self-loops
  Wrote 3,253 edges (333 TFs, 68 targets)
    TF→TF:     8
    TF→non-TF: 3,245

=== CD4_act ===
  GRN matrix: (4182, 4182), weight range [0.00001, 0.06426]
  TFs flagged in .var[isTF]: 263 / 4182
  Removed 6 self-loops
  Wrote 2,200 edges (263 TFs, 57 targets)
    TF→TF:     8
    TF→non-TF: 2,192

=== CD8_rest ===
  GRN matrix: (3487, 3487), weight range [0.00001, 0.07347]
  TFs flagged in .var[isTF]: 301 / 3487
  Removed 7 self-loops
  Wrote 3,084 edges (301 TFs, 59 targets)
    TF→TF:     94
    TF→non-TF: 2,990

=== CD8_act ===
  GRN matrix: (3368, 3368), weight range [0.00001, 0.06419]
  TFs flagged in .var[isTF]: 181 / 3368
  Removed 3 self-loops
  Wrote 1,635 edges (181 TFs, 39 targets)
    TF→TF:     28
    TF→non-TF: 1,607

Combined: 10,172 total edges → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revised_fi

## Cell 4 — Before vs after comparison

Loads the original (buggy) CSVs alongside the new (fixed) ones and shows the diff.
This is the table to bring to your PI.

In [17]:
rows = []
for state in STATES:
    old_csv = f'{GRN_IN_DIR}/{state}_scprint_pi_revised_edges.csv'
    new_csv = f'{GRN_OUT_DIR}/{state}_scprint_pi_revised_fixed_edges.csv'
    old = pd.read_csv(old_csv) if os.path.exists(old_csv) else pd.DataFrame(columns=['TF','target'])
    new = pd.read_csv(new_csv)

    rows.append({
        'state': state,
        'tfs_in_old_csv':    old['TF'].nunique() if len(old) else 0,
        'tfs_in_new_csv':    new['TF'].nunique(),
        'edges_in_old_csv':  len(old),
        'edges_in_new_csv':  len(new),
        'tfs_recovered':     new['TF'].nunique() - (old['TF'].nunique() if len(old) else 0),
    })

comparison = pd.DataFrame(rows)
print('=== Before/after comparison ===')
print(comparison.to_string(index=False))
print()

# Show a few TFs that were dropped by the bug, to make the recovery concrete
print('=== Spot-check: TFs recovered in CD4_act ===')
state = 'CD4_act'
old = pd.read_csv(f'{GRN_IN_DIR}/{state}_scprint_pi_revised_edges.csv')
new = pd.read_csv(f'{GRN_OUT_DIR}/{state}_scprint_pi_revised_fixed_edges.csv')
old_tfs = set(old['TF'])
new_tfs = set(new['TF'])
recovered = sorted(new_tfs - old_tfs)
lost = sorted(old_tfs - new_tfs)
print(f'  TFs in old CSV only ({len(lost)}): {lost}')
print(f'  TFs in new CSV only ({len(recovered)}): {recovered[:30]}'
      f'{"..." if len(recovered) > 30 else ""}')
print(f'  TFs in both: {len(old_tfs & new_tfs)}')

=== Before/after comparison ===
   state  tfs_in_old_csv  tfs_in_new_csv  edges_in_old_csv  edges_in_new_csv  tfs_recovered
CD4_rest               6             333               185              3253            327
 CD4_act               4             263               105              2200            259
CD8_rest               7             301               935              3084            294
 CD8_act               3             181               522              1635            178

=== Spot-check: TFs recovered in CD4_act ===
  TFs in old CSV only (0): []
  TFs in new CSV only (259): ['AHR', 'AKAP8', 'AKNA', 'ANKZF1', 'ARID5A', 'ARID5B', 'ATF1', 'ATF3', 'ATF4', 'ATF5', 'ATF6', 'ATF7', 'BATF', 'BATF3', 'BCL11B', 'BHLHE40', 'CAMTA1', 'CAMTA2', 'CARF', 'CDC5L', 'CEBPB', 'CEBPG', 'CEBPZ', 'CGGBP1', 'CHAMP1', 'CHCHD3', 'CLOCK', 'CREB1', 'CREB3', 'CREB3L2']...
  TFs in both: 4


## Cell 5 — Directionality sanity check

The original notebook claims `column = regulator, row = target` but this is an assumption
about how scPRINT2 stores its attention output, not something verified against the
scPRINT2 source. This cell checks the assumption against known T-cell biology.

**Test:** for well-established T-cell master regulators (BATF, IRF4, FOSB, TCF7, KLF2, FOXO1),
we expect their *outgoing* total weight (under our convention: sum over that TF's column)
to be substantially larger than their *incoming* total weight (sum over that TF's row).

**Interpretation:**
- If outgoing >> incoming → directionality is as documented (column = regulator). ✓
- If incoming >> outgoing → directionality is inverted; swap rows/cols in cell 3 and rerun.
- If outgoing ≈ incoming → the matrix is effectively symmetric (e.g., from mean-aggregated
  multi-head attention). Direction labels are arbitrary and the choice doesn't matter
  much for set-level overlap analyses, but flag this in the methods section.

In [18]:
KNOWN_ACTIVATION_TFS = ['BATF', 'IRF4', 'FOSB', 'JUN', 'JUNB', 'NFKB1', 'NFKB2', 'RELA', 'STAT3']
KNOWN_RESTING_TFS    = ['TCF7', 'KLF2', 'FOXO1', 'LEF1', 'BACH2', 'FOXP1']
PROBES               = KNOWN_ACTIVATION_TFS + KNOWN_RESTING_TFS

for state in STATES:
    grn = ad.read_h5ad(f'{GRN_IN_DIR}/{state}_grn.h5ad')
    mat = grn.varp['GRN']
    dense = mat.toarray() if sp.issparse(mat) else np.asarray(mat)
    symbols = grn.var['symbol'].values
    sym2idx = {s: i for i, s in enumerate(symbols)}

    print(f'\n--- {state} ---')
    print(f'  {"TF":<8}  {"outgoing (col sum)":>20}  {"incoming (row sum)":>20}  {"out/in ratio":>14}')
    ratios = []
    for tf in PROBES:
        if tf not in sym2idx:
            continue
        i = sym2idx[tf]
        outgoing = dense[:, i].sum()  # column sum = TF as regulator (under stated convention)
        incoming = dense[i, :].sum()  # row sum = TF as target
        ratio = outgoing / incoming if incoming > 0 else float('inf')
        ratios.append(ratio)
        print(f'  {tf:<8}  {outgoing:>20.4f}  {incoming:>20.4f}  {ratio:>14.3f}')

    if ratios:
        median_ratio = float(np.median(ratios))
        print(f'  Median out/in ratio across known TFs: {median_ratio:.3f}')
        if median_ratio > 1.5:
            print(f'  → Consistent with column=regulator convention.')
        elif median_ratio < 0.67:
            print(f'  → INVERTED: row appears to be the regulator side. Swap rows/cols in cell 3.')
        else:
            print(f'  → Approximately symmetric. Direction choice is largely arbitrary.')

    del grn, dense
    gc.collect()


--- CD4_rest ---
  TF          outgoing (col sum)    incoming (row sum)    out/in ratio
  BATF                    0.3094                1.0000           0.309
  IRF4                    1.2237                1.0000           1.224
  JUN                     0.2218                1.0000           0.222
  JUNB                    0.2979                1.0000           0.298
  NFKB1                   0.2960                1.0000           0.296
  NFKB2                   0.2629                1.0000           0.263
  RELA                    0.1961                1.0000           0.196
  STAT3                   0.2429                1.0000           0.243
  TCF7                    0.3498                1.0000           0.350
  KLF2                    1.3569                1.0000           1.357
  FOXO1                   0.3953                1.0000           0.395
  LEF1                    0.4891                1.0000           0.489
  BACH2                   1.9825                1.0000     

## Cell 6 — Top TFs per state (sanity check on biology)

Now that TF→TF edges are kept, the top TFs by outgoing weight should look biologically
sensible — activation TFs (BATF, IRF4, FOSB, JUN, NFKB) dominating in `_act` states,
resting TFs (TCF7, KLF2, FOXO1, LEF1) dominating in `_rest` states.

If the top TFs are still housekeeping/ribosomal regulators (GPBP1, PREB, SREBF1), that
would mean the scPRINT2 attention itself is tracking expression-level structure more than
regulatory signal — which is a separate (and known) limitation of attention-based GRN
methods, but worth flagging.

In [19]:
TOP_N = 20

for state, edges in all_edges.items():
    tf_scores = (
        edges.groupby('TF')['weight'].sum()
             .sort_values(ascending=False)
             .head(TOP_N)
    )
    print(f'\n[{state}] Top {TOP_N} TFs by total outgoing regulatory weight:')
    print(tf_scores.to_string())


[CD4_rest] Top 20 TFs by total outgoing regulatory weight:
TF
ZBED2      0.277696
ZNF598     0.277167
MAFK       0.270973
NRF1       0.263930
HIC1       0.262482
ZNF501     0.260179
ZNF101     0.257643
ZFP91      0.257094
ELF4       0.254726
ZSCAN18    0.251433
HIVEP3     0.250420
BACH1      0.249149
CSRNP1     0.248981
ZNF764     0.247405
ZNF274     0.247068
MYBL1      0.246413
NFATC2     0.246255
TBX21      0.246119
ZNF770     0.244999
MEF2A      0.244568

[CD4_act] Top 20 TFs by total outgoing regulatory weight:
TF
ZNF101     0.226485
SREBF1     0.224421
YY1        0.223286
ZNF598     0.218008
ZNF266     0.215905
ZNF277     0.213952
ZNF444     0.209876
GMEB1      0.209621
ZNF770     0.207227
ZNF397     0.206233
ZSCAN18    0.205040
HIC1       0.204627
PBX2       0.203861
ZFP91      0.198986
REL        0.198683
TRAFD1     0.198386
CRX        0.198261
ZNF580     0.197710
STAT5B     0.196877
IKZF2      0.196732

[CD8_rest] Top 20 TFs by total outgoing regulatory weight:
TF
ZNF444    0.

## What to do next

1. **Update your overlap analysis** to point at the new directory:
   ```python
   SCPRINT_EDGE_PATHS = {
       'CD4_rest': f'{DRIVE_ROOT}/scPRINT/grn_outputs_pi_revised_fixed/CD4_rest_scprint_pi_revised_fixed_edges.csv',
       'CD4_act':  f'{DRIVE_ROOT}/scPRINT/grn_outputs_pi_revised_fixed/CD4_act_scprint_pi_revised_fixed_edges.csv',
       'CD8_rest': f'{DRIVE_ROOT}/scPRINT/grn_outputs_pi_revised_fixed/CD8_rest_scprint_pi_revised_fixed_edges.csv',
       'CD8_act':  f'{DRIVE_ROOT}/scPRINT/grn_outputs_pi_revised_fixed/CD8_act_scprint_pi_revised_fixed_edges.csv',
   }
   ```
   and rerun the per-state overlap matrices (View D).

2. **Send the cell-4 comparison table to your PI** as the diff between the buggy and fixed
   extractions. The headline number is the jump from 3–7 TFs to (expected) 150–300 TFs per state.

3. **Check cell 5's directionality output.** If the out/in ratio is roughly 1.0 (symmetric),
   note it in your methods section. If it's < 0.67 (inverted), swap `rows` and `cols` in
   cell 3's DataFrame construction and rerun.

4. **Don't delete the old CSVs.** Keep `grn_outputs_pi_revisedv3/` around as the buggy
   baseline so the fix is auditable.

In [20]:
# Paths for the other two networks
PYSCENIC_RSS_PATH = f'{DRIVE_ROOT}/pyscenic/pruned_rss_score_matrix_final.csv'
PERTURB_EDGE_PATHS = {
    'Rest':     f'{DRIVE_ROOT}/perturbnet_outputs/perturb_net_tfs_only_Rest_filtered.csv',
    'Stim8hr':  f'{DRIVE_ROOT}/perturbnet_outputs/perturb_net_tfs_only_Stim8hr_filtered.csv',
    'Stim48hr': f'{DRIVE_ROOT}/perturbnet_outputs/perturb_net_tfs_only_Stim48hr_filtered.csv',
}

# pySCENIC: top-N regulons per cell state (robust to CD4 vs CD8 RSS scale differences)
PYSCENIC_TOP_N_PER_STATE = 50

# Output dir for overlap matrices (separate from buggy version's outputs)
DRIVE_ANALYSIS = f'{DRIVE_ROOT}/network_overlap_analysis_fixed'
os.makedirs(DRIVE_ANALYSIS, exist_ok=True)

# --- pySCENIC: load RSS matrix and extract top-N TFs per state ---
print('Loading pySCENIC RSS matrix...')
rss = pd.read_csv(PYSCENIC_RSS_PATH, index_col=0)
rss.columns = rss.columns.str.replace(r'\(\+\)$', '', regex=True)
print(f'  RSS shape: {rss.shape}, states: {list(rss.index)}')

pyscenic_state_tfs = {}
for s in ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']:
    if s not in rss.index:
        print(f'  WARNING: {s} missing from RSS matrix')
        continue
    top = rss.loc[s].nlargest(PYSCENIC_TOP_N_PER_STATE)
    pyscenic_state_tfs[s] = set(top.index)
    print(f'  pySCENIC {s}: top {len(top)} TFs (RSS range {top.min():.3f}-{top.max():.3f})')

# --- Perturb-seq: load each condition's filtered edges ---
print('\nLoading perturb-seq edges...')
perturb_edges = {}
for cond, path in PERTURB_EDGE_PATHS.items():
    df = pd.read_csv(path)
    tf_col = 'source_TF' if 'source_TF' in df.columns else df.columns[0]
    perturb_edges[cond] = df
    print(f'  Perturb-seq CD4 {cond}: {len(df):,} edges, {df[tf_col].nunique()} unique TFs')

# --- scPRINT2: load the FIXED CSVs from cell 3 ---
SCPRINT_FIXED_PATHS = {
    s: f'{GRN_OUT_DIR}/{s}_scprint_pi_revised_fixed_edges.csv'
    for s in STATES
}
print('\nLoading scPRINT2 fixed edges...')
scprint_edges = {}
for s, path in SCPRINT_FIXED_PATHS.items():
    df = pd.read_csv(path)
    scprint_edges[s] = df
    print(f'  scPRINT2 {s}: {len(df):,} edges, {df["TF"].nunique()} unique TFs')

Loading pySCENIC RSS matrix...
  RSS shape: (4, 235), states: ['CD4_act', 'CD8_rest', 'CD8_act', 'CD4_rest']
  pySCENIC CD4_rest: top 50 TFs (RSS range 0.447-0.522)
  pySCENIC CD4_act: top 50 TFs (RSS range 0.445-0.489)
  pySCENIC CD8_rest: top 50 TFs (RSS range 0.313-0.385)
  pySCENIC CD8_act: top 50 TFs (RSS range 0.298-0.368)

Loading perturb-seq edges...
  Perturb-seq CD4 Rest: 28,836 edges, 409 unique TFs
  Perturb-seq CD4 Stim8hr: 33,569 edges, 429 unique TFs
  Perturb-seq CD4 Stim48hr: 28,521 edges, 424 unique TFs

Loading scPRINT2 fixed edges...
  scPRINT2 CD4_rest: 3,253 edges, 333 unique TFs
  scPRINT2 CD4_act: 2,200 edges, 263 unique TFs
  scPRINT2 CD8_rest: 3,084 edges, 301 unique TFs
  scPRINT2 CD8_act: 1,635 edges, 181 unique TFs


In [21]:
records = []

# scPRINT2: per-state FIXED edges
for s, df in scprint_edges.items():
    lineage, condition = s.split('_')   # 'CD4', 'rest'
    for tf in df['TF'].dropna().unique():
        records.append({'tf': tf, 'method': 'scPRINT2',
                        'lineage': lineage, 'condition': condition})

# pySCENIC: per-state top-N from RSS
for s, tfs in pyscenic_state_tfs.items():
    lineage, condition = s.split('_')
    for tf in tfs:
        records.append({'tf': tf, 'method': 'pySCENIC',
                        'lineage': lineage, 'condition': condition})

# Perturb-seq: CD4 only, per condition (normalize labels)
cond_normalize = {'Rest': 'rest', 'Stim8hr': 'stim8hr', 'Stim48hr': 'stim48hr'}
for cond, df in perturb_edges.items():
    condition = cond_normalize[cond]
    tf_col = 'source_TF' if 'source_TF' in df.columns else df.columns[0]
    for tf in df[tf_col].dropna().unique():
        records.append({'tf': tf, 'method': 'Perturb-seq',
                        'lineage': 'CD4', 'condition': condition})

tf_long = pd.DataFrame(records)
tf_long['present'] = 1
print(f'Long-form table: {len(tf_long):,} rows, {tf_long["tf"].nunique():,} unique TFs')
print(f'\nPer-(method, lineage, condition) TF counts:')
print(tf_long.groupby(['method', 'lineage', 'condition']).size().rename('n_tfs').to_string())

tf_long.to_csv(f'{DRIVE_ANALYSIS}/tf_presence_long_fixed.csv', index=False)

Long-form table: 2,540 rows, 745 unique TFs

Per-(method, lineage, condition) TF counts:
method       lineage  condition
Perturb-seq  CD4      rest         409
                      stim48hr     424
                      stim8hr      429
pySCENIC     CD4      act           50
                      rest          50
             CD8      act           50
                      rest          50
scPRINT2     CD4      act          263
                      rest         333
             CD8      act          181
                      rest         301


In [22]:
# Flat {method_state_label -> TF set} dict
columns_dict = {}
for (method, lineage, condition), grp in tf_long.groupby(['method', 'lineage', 'condition']):
    columns_dict[f'{method}_{lineage}_{condition}'] = set(grp['tf'])

# Which method-state columns belong in each state's block
# CD4_act gets both perturb-seq timepoints since stim IS activation
state_blocks = {
    'CD4_rest': ['scPRINT2_CD4_rest', 'pySCENIC_CD4_rest',
                 'Perturb-seq_CD4_rest'],
    'CD4_act':  ['scPRINT2_CD4_act',  'pySCENIC_CD4_act',
                 'Perturb-seq_CD4_stim8hr', 'Perturb-seq_CD4_stim48hr'],
    'CD8_rest': ['scPRINT2_CD8_rest', 'pySCENIC_CD8_rest'],
    'CD8_act':  ['scPRINT2_CD8_act',  'pySCENIC_CD8_act'],
}

print('=' * 70)
print('View D (per-state): overlap and Jaccard matrices — FIXED scPRINT2')
print('=' * 70)

per_state_matrices = {}

for state, cols in state_blocks.items():
    cols_present = [c for c in cols if c in columns_dict]
    if not cols_present:
        print(f'\n  {state}: no matching columns, skipping')
        continue

    overlap = pd.DataFrame(0, index=cols_present, columns=cols_present, dtype=int)
    jaccard = pd.DataFrame(0.0, index=cols_present, columns=cols_present)
    for c1 in cols_present:
        for c2 in cols_present:
            inter = len(columns_dict[c1] & columns_dict[c2])
            union = len(columns_dict[c1] | columns_dict[c2])
            overlap.loc[c1, c2] = inter
            jaccard.loc[c1, c2] = round(inter / union, 4) if union else 0.0

    per_state_matrices[state] = {'overlap': overlap, 'jaccard': jaccard}

    print(f'\n  --- {state} ---')
    print(f'\n  Overlap counts (diagonal = total TFs in that method-state):')
    print(overlap.to_string())
    print(f'\n  Jaccard similarity:')
    print(jaccard.to_string())

    overlap.to_csv(f'{DRIVE_ANALYSIS}/overlap_{state}_fixed.csv')
    jaccard.to_csv(f'{DRIVE_ANALYSIS}/jaccard_{state}_fixed.csv')

print(f'\n{"=" * 70}')
print(f'Saved {len(per_state_matrices) * 2} files to:')
print(f'  {DRIVE_ANALYSIS}')
print('=' * 70)

View D (per-state): overlap and Jaccard matrices — FIXED scPRINT2

  --- CD4_rest ---

  Overlap counts (diagonal = total TFs in that method-state):
                      scPRINT2_CD4_rest  pySCENIC_CD4_rest  Perturb-seq_CD4_rest
scPRINT2_CD4_rest                   333                 39                   144
pySCENIC_CD4_rest                    39                 50                    17
Perturb-seq_CD4_rest                144                 17                   409

  Jaccard similarity:
                      scPRINT2_CD4_rest  pySCENIC_CD4_rest  Perturb-seq_CD4_rest
scPRINT2_CD4_rest                1.0000             0.1134                0.2408
pySCENIC_CD4_rest                0.1134             1.0000                0.0385
Perturb-seq_CD4_rest             0.2408             0.0385                1.0000

  --- CD4_act ---

  Overlap counts (diagonal = total TFs in that method-state):
                          scPRINT2_CD4_act  pySCENIC_CD4_act  Perturb-seq_CD4_stim8hr  Perturb-seq

In [23]:
# Diagnostic: did the fix actually do anything?
import pandas as pd
import os

GRN_OUT_DIR = '/content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revised_fixed'
GRN_IN_DIR  = '/content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3'

for state in ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']:
    old_path = f'{GRN_IN_DIR}/{state}_scprint_pi_revised_edges.csv'
    new_path = f'{GRN_OUT_DIR}/{state}_scprint_pi_revised_fixed_edges.csv'

    print(f'=== {state} ===')

    # Does the new file even exist?
    print(f'  Old exists: {os.path.exists(old_path)}')
    print(f'  New exists: {os.path.exists(new_path)}')

    if not os.path.exists(new_path):
        print(f'  → cell 3 never wrote this file. Re-run cell 3.')
        continue

    old = pd.read_csv(old_path)
    new = pd.read_csv(new_path)

    # File modification times — is the "new" file actually new?
    import datetime
    old_mtime = datetime.datetime.fromtimestamp(os.path.getmtime(old_path))
    new_mtime = datetime.datetime.fromtimestamp(os.path.getmtime(new_path))
    print(f'  Old file mtime: {old_mtime}')
    print(f'  New file mtime: {new_mtime}')

    # Are they identical?
    print(f'  Old: {len(old)} edges, {old["TF"].nunique()} TFs')
    print(f'  New: {len(new)} edges, {new["TF"].nunique()} TFs')

    # Does the new file have a target_is_tf column? (cell 3 added this)
    print(f'  New has target_is_tf column: {"target_is_tf" in new.columns}')
    if 'target_is_tf' in new.columns:
        print(f'  Of new edges, {new["target_is_tf"].sum()} are TF→TF '
              f'({100 * new["target_is_tf"].mean():.1f}%)')

    print()

=== CD4_rest ===
  Old exists: True
  New exists: True
  Old file mtime: 2026-05-01 23:09:55
  New file mtime: 2026-05-12 18:22:38
  Old: 185 edges, 6 TFs
  New: 3253 edges, 333 TFs
  New has target_is_tf column: True
  Of new edges, 8 are TF→TF (0.2%)

=== CD4_act ===
  Old exists: True
  New exists: True
  Old file mtime: 2026-05-01 23:09:55
  New file mtime: 2026-05-12 18:22:39
  Old: 105 edges, 4 TFs
  New: 2200 edges, 263 TFs
  New has target_is_tf column: True
  Of new edges, 8 are TF→TF (0.4%)

=== CD8_rest ===
  Old exists: True
  New exists: True
  Old file mtime: 2026-05-01 23:09:57
  New file mtime: 2026-05-12 18:22:39
  Old: 935 edges, 7 TFs
  New: 3084 edges, 301 TFs
  New has target_is_tf column: True
  Of new edges, 94 are TF→TF (3.0%)

=== CD8_act ===
  Old exists: True
  New exists: True
  Old file mtime: 2026-05-01 23:09:57
  New file mtime: 2026-05-12 18:22:39
  Old: 522 edges, 3 TFs
  New: 1635 edges, 181 TFs
  New has target_is_tf column: True
  Of new edges, 28 ar

In [24]:
import anndata as ad
import numpy as np

SCPRINT_DIR = '/content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3'
grn = ad.read_h5ad(f'{SCPRINT_DIR}/CD4_rest_grn.h5ad')
mat = grn.varp['GRN']
dense = np.asarray(mat.toarray() if hasattr(mat, 'toarray') else mat)
is_tf = grn.var['isTF'].astype(bool).values
print(f'GRN shape: {dense.shape}, TFs flagged: {is_tf.sum()}')
print()

conventions = [
    ('A) any-source -> any-target (no mask)',
     dense > 0.01),
    ('B) row=non-TF, col=TF [original mask]',
     (dense > 0.01) & np.outer(~is_tf, is_tf)),
    ('C) row=TF, col=non-TF [transposed of B]',
     (dense > 0.01) & np.outer(is_tf, ~is_tf)),
    ('D) any-row, col=TF [your cell-3 "fix"]',
     (dense > 0.01) & np.outer(np.ones_like(is_tf), is_tf)),
    ('E) row=TF, any-col [the opposite fix]',
     (dense > 0.01) & np.outer(is_tf, np.ones_like(is_tf))),
]

for name, mask in conventions:
    n_edges = int(mask.sum())
    cols_with_edges = np.where(mask.any(axis=0))[0]
    rows_with_edges = np.where(mask.any(axis=1))[0]
    tfs_as_cols = int(sum(is_tf[c] for c in cols_with_edges))
    tfs_as_rows = int(sum(is_tf[r] for r in rows_with_edges))
    print(f'{name}')
    print(f'  edges = {n_edges:,}')
    print(f'  TFs among unique cols-with-edges = {tfs_as_cols}')
    print(f'  TFs among unique rows-with-edges = {tfs_as_rows}')
    print()

GRN shape: (4251, 4251), TFs flagged: 333

A) any-source -> any-target (no mask)
  edges = 38,893
  TFs among unique cols-with-edges = 9
  TFs among unique rows-with-edges = 333

B) row=non-TF, col=TF [original mask]
  edges = 185
  TFs among unique cols-with-edges = 6
  TFs among unique rows-with-edges = 0

C) row=TF, col=non-TF [transposed of B]
  edges = 3,245
  TFs among unique cols-with-edges = 0
  TFs among unique rows-with-edges = 333

D) any-row, col=TF [your cell-3 "fix"]
  edges = 200
  TFs among unique cols-with-edges = 9
  TFs among unique rows-with-edges = 15

E) row=TF, any-col [the opposite fix]
  edges = 3,260
  TFs among unique cols-with-edges = 8
  TFs among unique rows-with-edges = 333

